In [1]:
import pandas as pd
import json
import re
import numpy as np
from sklearn.metrics import accuracy_score
from scipy.stats import kendalltau

In [2]:
GROUND_TRUTH_FILE = '../../data/judged/human_ground_truth.csv'
JUDGE_FILES = {
    'claude': '../../data/judged/sample_master_claude_judge.csv',
    'prometheus': '../../data/judged/sample_master_prometheus_judge.csv',
    'mistral': '../../data/judged/sample_master_mistral_judge.csv'
}

In [3]:
df_analysis = pd.read_csv(GROUND_TRUTH_FILE)
print(f"Initial DataFrame loaded with {len(df_analysis)} rows.")

Initial DataFrame loaded with 150 rows.


In [4]:
for judge_name, filename in JUDGE_FILES.items():
    try:
        df_judge = pd.read_csv(filename, usecols=['evaluation_id', f'evaluation_{judge_name}'])
        df_analysis = pd.merge(df_analysis, df_judge, on='evaluation_id', how='left')
        print(f"Data from judge '{judge_name}' successfully merged.")
    except Exception as e:
        print(f"ERROR processing the file for '{judge_name}': {e}")

print("\nShape of the DataFrame after merging:", df_analysis.shape)
print("Sample of raw data:")
df_analysis.head(3)

Data from judge 'claude' successfully merged.
Data from judge 'prometheus' successfully merged.
Data from judge 'mistral' successfully merged.

Shape of the DataFrame after merging: (150, 5)
Sample of raw data:


,evaluation_id,human_winner,evaluation_claude,evaluation_prometheus,evaluation_mistral
0,CG001_sabia-3.1_2_General Knowledge_detailed_e...,Tie,"{\n ""winner"": ""A"",\n ""general_justificat...","{\n""winner"": ""Tie"",\n""general_justification"": ...","{\n ""winner"": ""A"",\n ""general_j..."
1,CG002_1_General Knowledge_minimum_gemini-1.5-p...,B,"{\n ""winner"": ""A"",\n ""general_justificat...","Based on the analysis, the winner is ""A"".\n\nT...","{\n ""winner"": ""A"",\n ""general_j..."
2,CG002_2_General Knowledge_sabia-3.1_contextual...,B,"{\n ""winner"": ""B"",\n ""general_justificat...","{\n""winner"": ""Tie"",\n""general_justification"": ...","{\n ""winner"": ""B"",\n ""general_justification..."


### PROCESS CLAUDE EVALUATIONS

In [5]:
df_analysis['claude_winner'] = None
df_analysis['claude_total_score'] = np.nan

In [6]:
for index, row in df_analysis.iterrows():
    try:
        data = json.loads(row['evaluation_claude'])
        winner = data.get('winner')
        criteria = data.get('criteria', {})
        df_analysis.loc[index, 'claude_winner'] = winner

        scores = []
        if criteria:
            for crit_details in criteria.values():
                if 'score' in crit_details:
                    scores.append(crit_details['score'])
                elif 'score_a' in crit_details:
                    scores.append(crit_details['score_a'])
                elif 'score_b' in crit_details:
                    scores.append(crit_details['score_b'])
                elif 'score_A' in crit_details:
                    scores.append(crit_details['score_A'])
                elif 'score_B' in crit_details:
                    scores.append(crit_details['score_B'])

        if scores:
            df_analysis.loc[index, 'claude_total_score'] = np.mean(scores)
        else:
            df_analysis.loc[index, 'claude_total_score'] = 0
    except (TypeError, json.JSONDecodeError):
        df_analysis.loc[index, 'claude_winner'] = 'Parsing Error'
        df_analysis.loc[index, 'claude_total_score'] = 0

print("Claude processing completed.")
print("Sample results for Claude:")
df_analysis[['evaluation_id', 'claude_winner', 'claude_total_score']].head()

Claude processing completed.
Sample results for Claude:


,evaluation_id,claude_winner,claude_total_score
0,CG001_sabia-3.1_2_General Knowledge_detailed_e...,A,5.00
1,CG002_1_General Knowledge_minimum_gemini-1.5-p...,A,4.75
2,CG002_2_General Knowledge_sabia-3.1_contextual...,B,4.50
3,CG002_3_General Knowledge_sabia-3.1_detailed_v...,A,4.75
4,CG002_5_General Knowledge_minimum_gemini-1.5-p...,B,4.75


In [7]:
score_zero_count = (df_analysis['claude_total_score'] == 0).sum()
print(f"Count where 'claude_total_score' == 0: {score_zero_count}")

winner_nan_count = df_analysis['claude_winner'].isna().sum()
print(f"Count where 'claude_winner' is NaN: {winner_nan_count}")

parsing_error_count = (df_analysis['claude_winner'] == 'Parsing Error').sum()
print(f"Count where 'claude_winner' == 'Parsing Error': {parsing_error_count}")

Count where 'claude_total_score' == 0: 0
Count where 'claude_winner' is NaN: 0
Count where 'claude_winner' == 'Parsing Error': 0


### PROCESS MISTRAL EVALUATIONS

In [8]:
df_analysis['mistral_winner'] = None
df_analysis['mistral_total_score'] = np.nan

In [9]:
for index, row in df_analysis.iterrows():
    eval_text = str(row['evaluation_mistral'])
    
    json_list = []
    
    try:
        # ATTEMPT 1: Try to read as a single valid JSON
        data = json.loads(eval_text)
        json_list.append(data)

    except (TypeError, json.JSONDecodeError):
        try:
            # ATTEMPT 2: If the first fails, fix text using a SPACE separator
            fixed_text = '[' + re.sub(r'\}\s+\{', '}, {', eval_text) + ']'
            json_list = json.loads(fixed_text)
        except (TypeError, json.JSONDecodeError):
            try:
                # ATTEMPT 3: Split JSONs using a more specific pattern
                # Look for patterns like "} [RESPONSE B] {" or "}\n\n{"
                fixed_text = re.sub(r'\}\s*\[RESPONSE\s+[A-Z]\]\s*\{', '}, {', eval_text, flags=re.IGNORECASE)
                fixed_text = '[' + re.sub(r'\}\s*\n\s*\{', '}, {', fixed_text) + ']'
                json_list = json.loads(fixed_text)
            except (TypeError, json.JSONDecodeError):
                try:
                    # ATTEMPT 4: Use a more generic JSON separator
                    fixed_text = '[' + re.sub(r'\}\s*\{', '}, {', eval_text) + ']'
                    json_list = json.loads(fixed_text)
                except (TypeError, json.JSONDecodeError):
                    try:
                        # ATTEMPT 5: Extract valid JSONs using brace counting
                        json_objects = []
                        i = 0
                        
                        while i < len(eval_text):
                            if eval_text[i] == '{':
                                brace_count = 0
                                start = i
                                in_string = False
                                escape = False
                                
                                while i < len(eval_text):
                                    char = eval_text[i]
                                    
                                    if char == '"' and not escape:
                                        in_string = not in_string
                                    elif char == '\\' and not escape:
                                        escape = True
                                        i += 1
                                        continue
                                    
                                    if not in_string:
                                        if char == '{':
                                            brace_count += 1
                                        elif char == '}':
                                            brace_count -= 1
                                            if brace_count == 0:
                                                json_str = eval_text[start:i+1]
                                                try:
                                                    obj = json.loads(json_str)
                                                    if 'winner' in obj and 'criteria' in obj:
                                                        json_objects.append(obj)
                                                except json.JSONDecodeError:
                                                    pass
                                                break
                                    
                                    escape = False
                                    i += 1
                            i += 1
                        
                        if json_objects:
                            json_list = json_objects
                        else:
                            raise json.JSONDecodeError("No valid JSON found", eval_text, 0)
                            
                    except (TypeError, json.JSONDecodeError):
                        try:
                            # ATTEMPT 6: Handle truncated JSON by finding last valid one
                            json_objects = []
                            
                            for end_pos in range(len(eval_text) - 1, -1, -1):
                                if eval_text[end_pos] == '}':
                                    test_text = eval_text[:end_pos + 1]
                                    
                                    brace_count = 0
                                    for start_pos in range(end_pos, -1, -1):
                                        if test_text[start_pos] == '}':
                                            brace_count += 1
                                        elif test_text[start_pos] == '{':
                                            brace_count -= 1
                                            
                                            if brace_count == 0:
                                                json_str = test_text[start_pos:end_pos + 1]
                                                try:
                                                    obj = json.loads(json_str)
                                                    if isinstance(obj, dict) and 'winner' in obj and 'criteria' in obj:
                                                        json_objects.append(obj)
                                                        break
                                                except json.JSONDecodeError:
                                                    continue
                            
                            if json_objects:
                                json_list = json_objects
                            else:
                                raise json.JSONDecodeError("No valid JSON found", eval_text, 0)
                                
                        except (TypeError, json.JSONDecodeError):
                            try:
                                # ATTEMPT 7: Fix unbalanced braces in truncated JSON
                                open_braces = eval_text.count('{')
                                close_braces = eval_text.count('}')
                                if open_braces > close_braces:
                                    fixed_text = eval_text + ('}' * (open_braces - close_braces))
                                elif close_braces > open_braces:
                                    fixed_text = eval_text[:-(close_braces - open_braces)]
                                else:
                                    fixed_text = eval_text

                                data = json.loads(fixed_text)
                                if isinstance(data, dict) and 'winner' in data and 'criteria' in data:
                                    json_list.append(data)
                                elif isinstance(data, list):
                                    json_list.extend(data)

                            except (TypeError, json.JSONDecodeError):
                                try:
                                    # ATTEMPT 8: Final fallback — fix broken quotes inside JSON strings
                                    fixed_text = eval_text

                                    # Escape internal double quotes
                                    fixed_text = re.sub(r'(?<!\\)"(?![:,}\]\s])', r'\\"', fixed_text)
                                    # Fix duplicated quotes
                                    fixed_text = re.sub(r'\\"{2,}', r'\\"', fixed_text)
                                    fixed_text = re.sub(r'""', '"', fixed_text)
                                    # Remove invalid commas between quotes
                                    fixed_text = re.sub(r'"\s*,\s*"', ' ', fixed_text)
                                    # Remove unnecessary line breaks
                                    fixed_text = fixed_text.replace('\n', ' ').replace('\r', ' ')
                                    # Balance braces
                                    open_braces = fixed_text.count('{')
                                    close_braces = fixed_text.count('}')
                                    if open_braces > close_braces:
                                        fixed_text += '}' * (open_braces - close_braces)
                                    # Balance quotes
                                    double_quotes = fixed_text.count('"')
                                    if double_quotes % 2 != 0:
                                        fixed_text += '"'

                                    # Final parse attempt
                                    data = json.loads(fixed_text)

                                    if isinstance(data, dict) and 'winner' in data and 'criteria' in data:
                                        json_list.append(data)
                                    elif isinstance(data, list):
                                        json_list.extend(data)

                                except (TypeError, json.JSONDecodeError) as e:
                                    print(f"Final failure at index {index}: {e}")
                                    df_analysis.loc[index, 'mistral_winner'] = 'Parsing Error'
                                    df_analysis.loc[index, 'mistral_total_score'] = 0
                                    continue

    results = []
    
    for data in json_list:
        winner = data.get('winner')
        criteria = data.get('criteria', {})
        
        scores = []
        if criteria:
            for crit_details in criteria.values():
                score = (
                    crit_details.get('score') or 
                    crit_details.get('score_A') or 
                    crit_details.get('score_a') or 
                    crit_details.get('score_B') or 
                    crit_details.get('score_b')
                )
                if score is not None:
                    try:
                        scores.append(float(score))
                    except (ValueError, TypeError):
                        continue
        
        avg_score = np.mean(scores) if scores else 0
        
        if winner and avg_score > 0:
            results.append((winner, avg_score))

    # Select the result with the highest score
    if results:
        best_winner, best_score = max(results, key=lambda item: item[1])
        df_analysis.loc[index, 'mistral_winner'] = best_winner
        df_analysis.loc[index, 'mistral_total_score'] = best_score
    else:
        df_analysis.loc[index, 'mistral_winner'] = 'Parsing Error (No valid JSON found)'
        df_analysis.loc[index, 'mistral_total_score'] = 0


print("Mistral processing completed.")
print("\nSample results for Mistral:")
df_analysis[['evaluation_id', 'mistral_winner', 'mistral_total_score']].head(5)

Mistral processing completed.

Sample results for Mistral:


,evaluation_id,mistral_winner,mistral_total_score
0,CG001_sabia-3.1_2_General Knowledge_detailed_e...,A,4.75
1,CG002_1_General Knowledge_minimum_gemini-1.5-p...,A,4.75
2,CG002_2_General Knowledge_sabia-3.1_contextual...,B,4.75
3,CG002_3_General Knowledge_sabia-3.1_detailed_v...,A,4.75
4,CG002_5_General Knowledge_minimum_gemini-1.5-p...,A,4.75


In [10]:
score_zero_count = (df_analysis['mistral_total_score'] == 0).sum()
print(f"Count where 'mistral_total_score' == 0: {score_zero_count}")

winner_nan_count = df_analysis['mistral_winner'].isna().sum()
print(f"Count where 'mistral_winner' is NaN: {winner_nan_count}")

parsing_error_count = (df_analysis['mistral_winner'] == 'Parsing Error').sum()
print(f"Count where 'mistral_winner' == 'Parsing Error': {parsing_error_count}")

Count where 'mistral_total_score' == 0: 0
Count where 'mistral_winner' is NaN: 0
Count where 'mistral_winner' == 'Parsing Error': 0


### PROCESS PROMETHEUS EVALUATIONS

In [11]:
df_analysis['prometheus_winner'] = None
df_analysis['prometheus_total_score'] = np.nan

In [12]:
for index, row in df_analysis.iterrows():
    eval_text = str(row['evaluation_prometheus'])
    
    # --- Regex for "winner is..." format ---
    winner_match = re.search(r'winner is (?:Response )?\"?([AB]|Tie)\"?', eval_text, re.IGNORECASE)
    
    if winner_match:
        winner = winner_match.group(1).capitalize()
        df_analysis.loc[index, 'prometheus_winner'] = winner
        
        # Look for score
        score_match = re.search(r'with a score of (\d+)', eval_text, re.IGNORECASE)
        if score_match:
            df_analysis.loc[index, 'prometheus_total_score'] = int(score_match.group(1))
        else:
            # Try to extract scores from criteria in the text
            scores = []
            score_patterns = [
                r'"score":\s*(\d+)',
                r'score:\s*(\d+)',
                r'"score"\s*(\d+)',
                r'"logical_coherence":\s*(\d+)',
                r'"relevance_and_focus":\s*(\d+)',
                r'"accuracy_and_truthfulness":\s*(\d+)',
                r'"conciseness_and_clarity":\s*(\d+)',
                r'Logical Coherence:\s*(\d+)',
                r'Relevance and Focus:\s*(\d+)',
                r'Accuracy and Truthfulness:\s*(\d+)',
                r'Conciseness and Clarity:\s*(\d+)',
                r'\*\s*Logical Coherence:\s*(\d+)',
                r'\*\s*Relevance and Focus:\s*(\d+)',
                r'\*\s*Accuracy and Truthfulness:\s*(\d+)',
                r'\*\s*Conciseness and Clarity:\s*(\d+)',
            ]
            for pattern in score_patterns:
                matches = re.findall(pattern, eval_text, re.IGNORECASE)
                scores.extend([int(m) for m in matches])
            
            if not scores:
                criteria_mentioned = sum([
                    1 for crit in ['logical_coherence', 'relevance_and_focus', 
                                   'accuracy_and_truthfulness', 'conciseness_and_clarity']
                    if re.search(crit, eval_text, re.IGNORECASE)
                ])
                
                if criteria_mentioned > 0:
                    positive_words = len(re.findall(r'\b(comprehensive|detailed|coherent|well-structured|accurate|clear|concise|relevant|focused|logical)\b', eval_text, re.IGNORECASE))
                    negative_words = len(re.findall(r'\b(not|lacking|poor|weak|unclear|inaccurate|irrelevant|inconsistent)\b', eval_text, re.IGNORECASE))
                    
                    if positive_words > negative_words:
                        scores = [5] * criteria_mentioned
                    elif positive_words > 0:
                        scores = [4] * criteria_mentioned
                    else:
                        scores = [3] * criteria_mentioned
            
            if scores:
                df_analysis.loc[index, 'prometheus_total_score'] = np.mean(scores)
            else:
                df_analysis.loc[index, 'prometheus_total_score'] = 0
        
        continue

    # --- Try to extract winner from other formats ---
    alt_winner = re.search(r'(?:the|overall)\s+winner\s+is\s+\"?([AB])\"?', eval_text, re.IGNORECASE)
    if alt_winner:
        df_analysis.loc[index, 'prometheus_winner'] = alt_winner.group(1).upper()
        
        scores = []
        for pattern in score_patterns:
            matches = re.findall(pattern, eval_text, re.IGNORECASE)
            scores.extend([int(m) for m in matches])
        
        if not scores:
            criteria_mentioned = sum([
                1 for crit in ['logical_coherence', 'relevance_and_focus', 
                               'accuracy_and_truthfulness', 'conciseness_and_clarity']
                if re.search(crit, eval_text, re.IGNORECASE)
            ])
            
            if criteria_mentioned > 0:
                positive_words = len(re.findall(r'\b(comprehensive|detailed|coherent|well-structured|accurate|clear|concise|relevant|focused|logical)\b', eval_text, re.IGNORECASE))
                negative_words = len(re.findall(r'\b(not|lacking|poor|weak|unclear|inaccurate|irrelevant|inconsistent)\b', eval_text, re.IGNORECASE))
                
                if positive_words > negative_words:
                    scores = [5] * criteria_mentioned
                elif positive_words > 0:
                    scores = [4] * criteria_mentioned
                else:
                    scores = [3] * criteria_mentioned
        
        df_analysis.loc[index, 'prometheus_total_score'] = np.mean(scores) if scores else 0
        continue

    # --- JSON logic fallback ---
    try:
        fixed_text = eval_text.strip()
        json_match = re.search(r'\{[^{}]*"winner"[^{}]*\}', fixed_text, re.DOTALL)
        if json_match:
            try:
                data = json.loads(json_match.group())
                df_analysis.loc[index, 'prometheus_winner'] = data.get('winner', 'Parsing Error')
                
                criteria = data.get('criteria', {})
                if isinstance(criteria, dict):
                    scores = [
                        details.get('score') 
                        for details in criteria.values() 
                        if isinstance(details, dict) and details.get('score') is not None
                    ]
                    df_analysis.loc[index, 'prometheus_total_score'] = np.mean([int(s) for s in scores]) if scores else 0
                else:
                    df_analysis.loc[index, 'prometheus_total_score'] = 0
                continue
            except:
                pass
        
        fixed_text = re.sub(r'"\s*"criteria"', '", "criteria"', fixed_text)
        fixed_text = re.sub(r'\}\s*\{', '}, {', fixed_text)
        if not fixed_text.strip().startswith('{'):
            fixed_text = '{' + fixed_text.split('{', 1)[-1]
        if fixed_text.count('{') > 1 and not fixed_text.strip().startswith('['):
            fixed_text = '[' + fixed_text + ']'

        data = json.loads(fixed_text)
        if isinstance(data, list):
            data = data[0] if data else {}

        df_analysis.loc[index, 'prometheus_winner'] = data.get('winner', 'Parsing Error')
        criteria = data.get('criteria')
        if isinstance(criteria, dict):
            scores = [
                details.get('score') 
                for details in criteria.values() 
                if isinstance(details, dict) and details.get('score') is not None
            ]
            df_analysis.loc[index, 'prometheus_total_score'] = np.mean([int(s) for s in scores]) if scores else 0
        else:
            df_analysis.loc[index, 'prometheus_total_score'] = 0

    except Exception:
        # --- Final fallback, extract scores directly from text ---
        try:
            winner_fallback = re.search(r'winner\s+is\s+["\']?([AB])["\']?', eval_text, re.IGNORECASE)
            if winner_fallback:
                df_analysis.loc[index, 'prometheus_winner'] = winner_fallback.group(1).upper()
            else:
                df_analysis.loc[index, 'prometheus_winner'] = 'Parsing Error'
            
            scores = []
            for pattern in score_patterns + [r':\s*(\d+)\s*\([^)]*response']:
                matches = re.findall(pattern, eval_text, re.IGNORECASE)
                scores.extend([int(m) for m in matches])
            
            if not scores:
                criteria_mentioned = sum([
                    1 for crit in ['logical_coherence', 'relevance_and_focus', 
                                   'accuracy_and_truthfulness', 'conciseness_and_clarity']
                    if re.search(crit, eval_text, re.IGNORECASE)
                ])
                if criteria_mentioned > 0:
                    positive_words = len(re.findall(r'\b(comprehensive|detailed|coherent|well-structured|accurate|clear|concise|relevant|focused|logical)\b', eval_text, re.IGNORECASE))
                    negative_words = len(re.findall(r'\b(not|lacking|poor|weak|unclear|inaccurate|irrelevant|inconsistent)\b', eval_text, re.IGNORECASE))
                    if positive_words > negative_words:
                        scores = [5] * criteria_mentioned
                    elif positive_words > 0:
                        scores = [4] * criteria_mentioned
                    else:
                        scores = [3] * criteria_mentioned
            
            df_analysis.loc[index, 'prometheus_total_score'] = np.mean(scores) if scores else 0
            
        except Exception:
            df_analysis.loc[index, 'prometheus_winner'] = 'Parsing Error'
            df_analysis.loc[index, 'prometheus_total_score'] = 0

print("Prometheus processing completed.")
print("Sample results for Prometheus:")
df_analysis[['evaluation_id', 'prometheus_winner', 'prometheus_total_score']].head()

Prometheus processing completed.
Sample results for Prometheus:


,evaluation_id,prometheus_winner,prometheus_total_score
0,CG001_sabia-3.1_2_General Knowledge_detailed_e...,Tie,3.75
1,CG002_1_General Knowledge_minimum_gemini-1.5-p...,A,5.00
2,CG002_2_General Knowledge_sabia-3.1_contextual...,Tie,4.00
3,CG002_3_General Knowledge_sabia-3.1_detailed_v...,A,4.75
4,CG002_5_General Knowledge_minimum_gemini-1.5-p...,B,4.75


In [13]:
score_zero_count = (df_analysis['prometheus_total_score'] == 0).sum()
print(f"Count where 'prometheus_total_score' == 0: {score_zero_count}")

winner_nan_count = df_analysis['prometheus_winner'].isna().sum()
print(f"Count where 'prometheus_winner' is NaN: {winner_nan_count}")

parsing_error_count = (df_analysis['prometheus_winner'] == 'Parsing Error').sum()
print(f"Count where 'prometheus_winner' == 'Parsing Error': {parsing_error_count}")

Count where 'prometheus_total_score' == 0: 0
Count where 'prometheus_winner' is NaN: 0
Count where 'prometheus_winner' == 'Parsing Error': 0


### CLEANING AND FINAL CHECK

In [14]:
columns_to_remove = ['evaluation_claude', 'evaluation_prometheus', 'evaluation_mistral']
df_final = df_analysis.drop(columns=columns_to_remove)

print("Original evaluation columns removed.")
print("Final DataFrame shape:", df_final.shape)
print("Sample of the cleaned DataFrame ready for analysis:")
df_final.head()

Original evaluation columns removed.
Final DataFrame shape: (150, 8)
Sample of the cleaned DataFrame ready for analysis:


,evaluation_id,human_winner,claude_winner,claude_total_score,mistral_winner,mistral_total_score,prometheus_winner,prometheus_total_score
0,CG001_sabia-3.1_2_General Knowledge_detailed_e...,Tie,A,5.00,A,4.75,Tie,3.75
1,CG002_1_General Knowledge_minimum_gemini-1.5-p...,B,A,4.75,A,4.75,A,5.00
2,CG002_2_General Knowledge_sabia-3.1_contextual...,B,B,4.50,B,4.75,Tie,4.00
3,CG002_3_General Knowledge_sabia-3.1_detailed_v...,A,A,4.75,A,4.75,A,4.75
4,CG002_5_General Knowledge_minimum_gemini-1.5-p...,Tie,B,4.75,A,4.75,B,4.75


### AGREEMENT ANALYSIS

In [17]:
def get_score_diff(row, judge_name):
    winner = row[f'{judge_name}_winner']
    score = row[f'{judge_name}_total_score']
    if winner == 'A': 
        return score
    if winner == 'B': 
        return -score
    return 0

print("="*50)
print("AGREEMENT ANALYSIS (JUDGE vs. HUMANS)")
print("="*50)

print("\n--- Paired Accuracy ---")
for judge in ['claude', 'prometheus', 'mistral']:
    df_valid = df_final.dropna(subset=['human_winner', f'{judge}_winner'])
    df_valid = df_valid[df_valid[f'{judge}_winner'] != 'Parsing Error']
    accuracy = accuracy_score(df_valid['human_winner'], df_valid[f'{judge}_winner'])
    print(f"  - {judge.title()} Accuracy: {accuracy:.2%} ({len(df_valid)} valid pairs)")

print("\n--- Kendall's Tau (Ranking Correlation) ---")
mapping = {'A': 1, 'B': -1, 'Tie': 0}
df_final['human_choice_numeric'] = df_final['human_winner'].map(mapping)

for judge in ['claude', 'prometheus', 'mistral']:
    score_diff = df_final.apply(lambda row: get_score_diff(row, judge), axis=1)
    
    valid_indices = df_final['human_choice_numeric'].notna() & score_diff.notna()
    human_numeric = df_final.loc[valid_indices, 'human_choice_numeric']
    judge_scores = score_diff[valid_indices]
    
    tau, p_value = kendalltau(human_numeric, judge_scores)
    print(f"  - {judge.title()} Kendall's Tau: {tau:.4f} (p-value: {p_value:.3g}) ({len(human_numeric)} valid pairs)")

# Save final results
df_final.to_csv('../../data/judged/final_consolidated_analysis.csv', index=False)
print("\nAgreement analysis complete. Results saved in 'final_consolidated_analysis.csv'.")

AGREEMENT ANALYSIS (JUDGE vs. HUMANS)

--- Paired Accuracy ---
  - Claude Accuracy: 47.33% (150 valid pairs)
  - Prometheus Accuracy: 35.33% (150 valid pairs)
  - Mistral Accuracy: 40.00% (150 valid pairs)

--- Kendall's Tau (Ranking Correlation) ---
  - Claude Kendall's Tau: 0.1773 (p-value: 0.00924) (150 valid pairs)
  - Prometheus Kendall's Tau: 0.1306 (p-value: 0.0675) (150 valid pairs)
  - Mistral Kendall's Tau: 0.0794 (p-value: 0.263) (150 valid pairs)

Agreement analysis complete. Results saved in 'final_consolidated_analysis.csv'.
